# CIFAR-10 + DINOv2 - completing the grid (N=1000)

Fills the one remaining hole in the vision block: CIFAR-10 has CLIP only, while
Fashion-MNIST has both CLIP and DINOv2. After this run every dataset is fully
crossed with its domain's encoders and no encoder appears only once.

**Protocol is identical to the 24 existing configurations:**
DINOv2 CLS token, L2 normalized -> PCA to 8 (fit on train) -> MinMax to [0, pi]
(fit on train, test clipped) -> N=1000 train / 200 test, seed 42 -> three
classical SVMs (linear, RBF gamma=scale, poly d=3 coef0=1, all C=1) -> IQP
1-layer 8-qubit quantum kernel -> SVC(precomputed, C=1) -> binary F1.

**Runtime:** ~10 min embeddings + ~83 min kernel + ~2 min diagnostics.
Every artifact is saved to Drive the moment it exists.

In [ ]:
!pip install pennylane pennylane-lightning transformers datasets torchvision --quiet

import pennylane as qml
import numpy as np
import torch, time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
warnings.filterwarnings('ignore')

print(f"PennyLane {qml.__version__} | torch {torch.__version__}")
DEVICE = qml.device('lightning.qubit', wires=8)
SEED = 42

---
## Step 0: Drive storage

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
TAG = 'cifar10_dinov2'
print(f"Save dir: {SAVE_DIR} | tag: {TAG}")

def _persist(path):
    dst = os.path.join(SAVE_DIR, os.path.basename(path))
    if os.path.abspath(path) != os.path.abspath(dst):
        shutil.copy(path, dst)
    print(f"  saved -> {dst}")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist(filename)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: CIFAR-10, airplane vs automobile

Same two classes and same sample sizes as the CIFAR-10 + CLIP configuration.

In [ ]:
# CIFAR-10 loader. Newer `datasets` requires the namespaced repo id;
# falls back to torchvision if the Hub is unavailable.
CLASS_A, CLASS_B = 0, 1          # airplane, automobile
N_TRAIN, N_TEST = 1000, 200

def balanced_indices(labels, n_total, seed):
    labels = np.asarray(labels)
    rng = np.random.RandomState(seed)
    idx_a = np.where(labels == CLASS_A)[0]; rng.shuffle(idx_a)
    idx_b = np.where(labels == CLASS_B)[0]; rng.shuffle(idx_b)
    half = n_total // 2
    idx = np.concatenate([idx_a[:half], idx_b[:half]])
    rng.shuffle(idx)
    return idx

train_imgs = test_imgs = None
try:
    from datasets import load_dataset
    ds = load_dataset('uoft-cs/cifar10')          # namespaced id
    IMG_COL = 'img' if 'img' in ds['train'].column_names else 'image'
    tr_idx = balanced_indices(ds['train']['label'], N_TRAIN, SEED)
    te_idx = balanced_indices(ds['test']['label'],  N_TEST,  SEED)
    tr_sel = ds['train'].select(tr_idx); te_sel = ds['test'].select(te_idx)
    train_imgs, y_tr = tr_sel[IMG_COL], np.array(tr_sel['label'])
    test_imgs,  y_te = te_sel[IMG_COL], np.array(te_sel['label'])
    print("loaded via datasets (uoft-cs/cifar10)")
except Exception as e:
    print(f"datasets failed ({type(e).__name__}: {e}); falling back to torchvision")
    from torchvision.datasets import CIFAR10
    tr_ds = CIFAR10(root='./cifar', train=True,  download=True)
    te_ds = CIFAR10(root='./cifar', train=False, download=True)
    tr_idx = balanced_indices(tr_ds.targets, N_TRAIN, SEED)
    te_idx = balanced_indices(te_ds.targets, N_TEST,  SEED)
    train_imgs = [tr_ds[int(i)][0] for i in tr_idx]
    test_imgs  = [te_ds[int(i)][0] for i in te_idx]
    y_tr = np.array([tr_ds.targets[int(i)] for i in tr_idx])
    y_te = np.array([te_ds.targets[int(i)] for i in te_idx])
    print("loaded via torchvision")

print(f"train {len(train_imgs)} imgs, balance {np.bincount(y_tr)}")
print(f"test  {len(test_imgs)} imgs, balance {np.bincount(y_te)}")
assert len(train_imgs) == N_TRAIN and len(test_imgs) == N_TEST
assert set(np.unique(y_tr)) == {CLASS_A, CLASS_B}

---
## Step 2: DINOv2 embeddings (CLS token, L2 normalized)

Matches the Fashion-MNIST + DINOv2 configuration exactly: `facebook/dinov2-base`,
class token from the final hidden state, L2 normalized.

In [ ]:
from transformers import AutoImageProcessor, AutoModel

proc  = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = AutoModel.from_pretrained('facebook/dinov2-base').eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(dev)
print(f"DINOv2 on {dev}")

@torch.no_grad()
def embed(images, batch=64):
    out = []
    for i in range(0, len(images), batch):
        chunk = [im.convert('RGB') for im in images[i:i+batch]]
        inputs = proc(images=chunk, return_tensors='pt').to(dev)
        h = model(**inputs).last_hidden_state[:, 0, :]        # CLS token
        h = h / torch.linalg.norm(h, dim=1, keepdim=True)     # L2 normalize
        out.append(h.cpu().numpy())
        if i % (batch*4) == 0: print(f"  {i+len(chunk)}/{len(images)}")
    return np.vstack(out)

t0 = time.time()
E_tr = embed(train_imgs)
E_te = embed(test_imgs)
print(f"embeddings {E_tr.shape}, {E_te.shape} in {(time.time()-t0)/60:.1f} min")
assert E_tr.shape[1] == 768, "expected dinov2-base 768-dim"


---
## Step 3: PCA to 8 dims, then MinMax to [0, pi]

Both fitted on train only; test values clipped to the training interval.

In [ ]:
pca = PCA(n_components=8, random_state=SEED).fit(E_tr)
P_tr, P_te = pca.transform(E_tr), pca.transform(E_te)
print(f"variance explained by 8 PCs: {pca.explained_variance_ratio_.sum():.4f}")

scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(P_tr)
X_tr = scaler.transform(P_tr)
X_te = np.clip(scaler.transform(P_te), 0, np.pi)

print(f"X_tr {X_tr.shape} range [{X_tr.min():.4f}, {X_tr.max():.4f}]")
print(f"X_te {X_te.shape} range [{X_te.min():.4f}, {X_te.max():.4f}]")
assert X_tr.shape == (1000, 8) and X_te.shape == (200, 8)
assert X_tr.min() >= 0 and X_tr.max() <= np.pi + 1e-9

checkpoint(f'{TAG}_embeddings.npz',
           lambda p: np.savez(p, X_tr_subset=X_tr, y_tr_subset=y_tr,
                              X_te_subset=X_te, y_te_subset=y_te))

---
## Step 4: Classical baselines (identical settings to the grid)

In [ ]:
def bf1(yt, yp): return float(f1_score(yt, yp, average='binary'))

baselines = {}
for name, clf in [
    ('Linear',     SVC(kernel='linear', C=1.0, random_state=SEED)),
    ('RBF',        SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED)),
    ('Polynomial', SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale', random_state=SEED))]:
    yp = clf.fit(X_tr, y_tr).predict(X_te)
    baselines[name] = {'accuracy': float(accuracy_score(y_te, yp)), 'f1': bf1(y_te, yp)}
    print(f"  {name:<12} acc={baselines[name]['accuracy']:.4f}  f1={baselines[name]['f1']:.4f}")

best_classical_f1 = max(v['f1'] for v in baselines.values())
best_classical_nm = max(baselines, key=lambda k: baselines[k]['f1'])
print(f"\nbest classical: {best_classical_nm} at {best_classical_f1:.4f}")

checkpoint(f'{TAG}_baseline_results.pkl',
           lambda p: pickle.dump(baselines, open(p, 'wb')))

---
## Step 5: IQP quantum kernel (8 qubits, single layer)

In [ ]:
N_QUBITS = 8

def iqp_feature_map(x, wires):
    n = len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i, w in enumerate(wires): qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

@qml.qnode(DEVICE)
def kernel_circuit(x1, x2):
    iqp_feature_map(x1, wires=range(N_QUBITS))
    qml.adjoint(iqp_feature_map)(x2, wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))

quantum_kernel = lambda a, b: float(kernel_circuit(a, b)[0])

assert np.isclose(quantum_kernel(X_tr[0], X_tr[0]), 1.0, atol=1e-5), "K(x,x) must be 1"
t0 = time.time()
for _ in range(3): quantum_kernel(X_tr[0], X_tr[1])
ms = (time.time()-t0)/3*1000
print(f"sanity OK, {ms:.1f} ms/eval -> N=1000 matrix approx {1000*1000*ms/1000/60:.0f} min")

---
## Step 6: Kernel matrices (~83 min) - saved the moment they finish

In [ ]:
print("Computing quantum kernel matrices (N=1000). Do not close the tab.")
t0 = time.time()
K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
elapsed = time.time() - t0
print(f"done in {elapsed/60:.1f} min")

checkpoint(f'{TAG}_kernel_matrices.npz',
           lambda p: np.savez(p, K_train_quantum=K_train, K_test_quantum=K_test,
                              X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te))
print("Kernel matrices safe.")

---
## Step 7: Quantum SVM and the utility gap

In [ ]:
qsvm = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr)
y_pred = qsvm.predict(K_test)
acc_q = float(accuracy_score(y_te, y_pred))
f1_q  = bf1(y_te, y_pred)
gap   = f1_q - best_classical_f1

print("="*54)
print(f"CIFAR-10 + DINOv2 (N=1000)")
print(f"  quantum    acc={acc_q:.4f}  F1={f1_q:.4f}")
print(f"  best class F1={best_classical_f1:.4f} ({best_classical_nm})")
print(f"  utility gap Delta = {gap:+.4f}")
print("="*54)

---
## Step 8: Diagnostics - identical definitions to Section 3.5

Centered multiclass KTA (Cortes et al.), normalized effective rank, and the
exact Huang geometric difference with ridge-regularized inverse.

In [ ]:
def kta(K, y):
    y = np.asarray(y); n = len(y)
    Y = np.zeros((n, n))
    for c in np.unique(y):
        m = (y == c).astype(float); Y += np.outer(m, m)
    H = np.eye(n) - np.ones((n, n))/n
    Kc, Yc = H @ K @ H, H @ Y @ H
    return float(np.sum(Kc*Yc) / np.sqrt(np.sum(Kc*Kc) * np.sum(Yc*Yc)))

def concentration(K):
    mu = np.linalg.eigvalsh(K); mu = np.maximum(mu, 0)
    lam = mu / mu.sum()
    return float(np.exp(-np.sum(lam * np.log(lam + 1e-15))) / K.shape[0])

def geometric_difference_proxy(K_Q, K_C):
    # effective-dimension ratio used during development; appendix only
    eff = lambda K: (np.trace(K)**2) / (np.sum(K**2) + 1e-10)
    return float(eff(K_Q) / (eff(K_C) + 1e-10))

def geometric_difference_exact(K_Q, K_C, lam=1e-6):
    n = K_Q.shape[0]
    K_Q = K_Q * n / np.trace(K_Q)
    K_C = K_C * n / np.trace(K_C)
    w, V = np.linalg.eigh(K_Q); w = np.maximum(w, 0)
    sqrtKQ = (V * np.sqrt(w)) @ V.T
    Kc_inv = np.linalg.inv(K_C + lam * n * np.eye(n))
    return float(np.sqrt(max(np.linalg.eigvalsh(sqrtKQ @ Kc_inv @ sqrtKQ).max(), 0.0)))

K_rbf_diag = rbf_kernel(X_tr, X_tr, gamma=1.0/X_tr.shape[1])   # diagnostic RBF, gamma = 1/d

A_Q  = kta(K_train, y_tr)
A_C  = kta(K_rbf_diag, y_tr)
rho  = concentration(K_train)
g_ex = geometric_difference_exact(K_train, K_rbf_diag)
g_px = geometric_difference_proxy(K_train, K_rbf_diag)

off_q = K_train[~np.eye(len(K_train), dtype=bool)].mean()
off_c = K_rbf_diag[~np.eye(len(K_rbf_diag), dtype=bool)].mean()

print(f"  A_Q (quantum KTA)   = {A_Q:.4f}")
print(f"  A_C (classical KTA) = {A_C:.4f}")
print(f"  rho (concentration) = {rho:.4f}")
print(f"  g   (exact)         = {g_ex:.2f}")
print(f"  g   (proxy, eff-dim)= {g_px:.2f}")
print(f"  off-diagonal mean: quantum {off_q:.4f} | RBF {off_c:.4f}")
print("  lambda sweep:", {f"{l:.0e}": round(geometric_difference_exact(K_train, K_rbf_diag, l), 2)
                          for l in [1e-4, 1e-5, 1e-6, 1e-7]})

---
## Step 9: Bootstrap confidence interval on the gap (1,000 resamples)

In [ ]:
rng = np.random.RandomState(SEED)
best_clf = {'Linear':     SVC(kernel='linear', C=1.0, random_state=SEED),
            'RBF':        SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED),
            'Polynomial': SVC(kernel='poly', degree=3, coef0=1, C=1.0, gamma='scale',
                              random_state=SEED)}[best_classical_nm].fit(X_tr, y_tr)
yc_pred = best_clf.predict(X_te)

boot = []
n_te = len(y_te)
for _ in range(1000):
    idx = rng.randint(0, n_te, n_te)
    if len(np.unique(y_te[idx])) < 2: continue
    boot.append(bf1(y_te[idx], y_pred[idx]) - bf1(y_te[idx], yc_pred[idx]))
boot = np.array(boot)
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
excludes_zero = bool(ci_lo > 0 or ci_hi < 0)

print(f"  gap {gap:+.4f}  95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]  excludes zero: {excludes_zero}")
print(f"  ({len(boot)} valid resamples)")

---
## Step 9b: Learning curve (reuses the saved kernel, no new circuits)

In [ ]:
train_sizes = [50, 100, 200, 400, 700, 1000]
lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
print(f"{'N':>5}  {'Quantum':>9}  {'Linear':>9}  {'RBF':>9}")
for n in train_sizes:
    yp = SVC(kernel='precomputed', C=1.0).fit(K_train[:n, :n], y_tr[:n]).predict(K_test[:, :n])
    lc['Quantum (IQP)'].append(bf1(y_te, yp))
    for cname, ck in [('Linear','linear'), ('RBF','rbf')]:
        clf = SVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 'scale',
                  random_state=SEED).fit(X_tr[:n], y_tr[:n])
        lc[cname].append(bf1(y_te, clf.predict(X_te)))
    print(f"{n:>5}  {lc['Quantum (IQP)'][-1]:>9.4f}  {lc['Linear'][-1]:>9.4f}  {lc['RBF'][-1]:>9.4f}")

---
## Step 10: Save a merge-ready record

In [ ]:
record = {
    'tag': TAG, 'domain': 'vision', 'dataset': 'cifar10_binary', 'encoder': 'dinov2',
    'metric': 'binary', 'n_qubits': N_QUBITS, 'n_train': int(len(X_tr)), 'n_test': int(len(X_te)),
    'compute_time_min': round(elapsed/60, 1),
    'quantum_accuracy': acc_q, 'quantum_f1': f1_q,
    'best_classical_f1': best_classical_f1, 'best_classical_kernel': best_classical_nm,
    'gap': gap,
    'kta': A_Q, 'c_kta': A_C, 'concentration': rho, 'geometric_difference_exact': g_ex, 'geometric_difference_proxy': g_px,
    'offdiag_quantum': float(off_q), 'offdiag_rbf': float(off_c),
    'ci_lo': float(ci_lo), 'ci_hi': float(ci_hi), 'excludes_zero': excludes_zero,
    'classical_baselines': baselines,
    'learning_curve_sizes': train_sizes,
    'learning_curve_results': lc,
}
checkpoint(f'{TAG}_grid_record.json', lambda p: json.dump(record, open(p, 'w'), indent=2))

print(json.dumps({k: v for k, v in record.items() if k != 'classical_baselines'}, indent=2))
print("\nUpload cifar10_dinov2_grid_record.json to continue the paper.")